## Installs and Initialization

In [1]:
!pip install "timesfm[torch,xreg]" mlflow dagshub -q  # xreg extra added for the covariate-comparison run below

import os
import time
import numpy as np
import pandas as pd
import mlflow
import dagshub
import torch
import timesfm

# Initialize MLflow
dagshub.init(repo_owner='ejoba22', repo_name='walmart-sales-forecasting', mlflow=True)
mlflow.set_experiment("TimesFM_Training")

print("Tracking URI:", mlflow.get_tracking_uri())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 758.2 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 35.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 47.5 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=c1f21aca-0ae2-4699-92e8-3c33a8475fa5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e78b3de9f29b08cb8d1496673be6eacfb060f20125ee9306a144c5f415b4a07d




Accessing as ejoba22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

Tracking URI: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow


## Data Loading & Formatting

In [15]:
BASE = '/kaggle/input/datasets/elenejobava/walmart-features-engineered/'
train_fe = pd.read_parquet(BASE + 'train_features.parquet')
train_fe["Date"] = pd.to_datetime(train_fe["Date"])

TRAIN_WEEKS = 143
CV_FOLD_HORIZON = 13
val_start_week = TRAIN_WEEKS - CV_FOLD_HORIZON

active_pairs = sorted(set(zip(train_fe.Store, train_fe.Dept)))
full_date_range = pd.date_range(train_fe["Date"].min(), periods=TRAIN_WEEKS, freq="7D")

records = []  # one tuple per pair, built in a single pass - can't drift apart

for store, dept in active_pairs:
    series = train_fe[(train_fe.Store == store) & (train_fe.Dept == dept)][['Date', 'Weekly_Sales', 'IsHoliday']]
    series = series.set_index('Date').reindex(full_date_range)

    sales = series['Weekly_Sales'].fillna(0).clip(lower=0).values.astype(np.float32)
    is_holiday = series['IsHoliday'].fillna(False).values.astype(bool)

    ctx = sales[:val_start_week]
    act = sales[val_start_week:val_start_week + CV_FOLD_HORIZON]
    w = np.where(is_holiday[val_start_week:val_start_week + CV_FOLD_HORIZON], 5.0, 1.0)
    hol = is_holiday.astype(np.int32)

    records.append((ctx, act, w, hol))

# Unzip - guaranteed identical length for all four, by construction
context_data, actuals, weights, holiday_full = map(list, zip(*records))

assert len(active_pairs) == len(context_data) == len(actuals) == len(weights) == len(holiday_full)
print(f"OK: {len(active_pairs)} series loaded consistently.")

OK: 3331 series loaded consistently.


## Model Loading & Holdout Validation (Zero-Shot)

In [3]:
torch.set_float32_matmul_precision("high")

print("Downloading and loading TimesFM 2.5 model...")
tfm_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

CONTEXT_LEN = 256  # >= 130 weeks of actual context, no silent truncation

tfm_model.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT_LEN,
        max_horizon=CV_FOLD_HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

with mlflow.start_run(run_name="TimesFM_ZeroShot_Holdout"):
    mlflow.set_tags({"model_type": "TimesFM", "run_type": "zero-shot-eval"})

    start_time = time.time()

    print("Forecasting...")
    point_fc, quant_fc = tfm_model.forecast(horizon=CV_FOLD_HORIZON, inputs=context_data)

    # infer_is_positive already keeps forecasts non-negative,
    # but clip is kept as a safety net for numerical edge cases
    preds = np.clip(point_fc, 0, None)
    elapsed = time.time() - start_time

    abs_err = np.abs(preds - np.array(actuals))
    wmae = np.sum(np.array(weights) * abs_err) / np.sum(np.array(weights))

    mlflow.log_metric("val_wmae", wmae)
    mlflow.log_metric("inference_time_minutes", elapsed / 60)
    mlflow.log_param("context_len", CONTEXT_LEN)
    mlflow.log_param("actual_context_weeks", val_start_week)
    mlflow.log_param("horizon", CV_FOLD_HORIZON)
    mlflow.log_param("normalize_inputs", True)
    mlflow.log_param("use_continuous_quantile_head", True)
    mlflow.log_param("infer_is_positive", True)
    mlflow.log_param("force_flip_invariance", True)
    mlflow.log_param("fix_quantile_crossing", True)

    print(f"TimesFM Zero-Shot WMAE: {wmae:.2f}")
    print(f"Inference completed in {elapsed/60:.2f} minutes!")

config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

Forecasting...
TimesFM Zero-Shot WMAE: 1156.72
Inference completed in 2.62 minutes!
🏃 View run TimesFM_ZeroShot_Holdout at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12/runs/01ffed78631f4da682c8b1150bdfe166
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12


In [10]:
assert len(active_pairs) == len(context_data) == len(actuals) == len(weights) == len(holiday_full), (
    f"Length mismatch! active_pairs={len(active_pairs)}, context_data={len(context_data)}, "
    f"actuals={len(actuals)}, weights={len(weights)}, holiday_full={len(holiday_full)}. "
    "This means the data-loading cell was re-run partially or out of order in this kernel "
    "session. Restart the kernel and Run All from the top."
)
print(f"OK: {len(active_pairs)} series loaded consistently.")

OK: 3331 series loaded consistently.


## Run 2: Effect of Context Length (Zero-Shot)

In [11]:
import math

SHORT_CONTEXT_LEN = 64  # ~1 year of weekly history, rounded up to a multiple of 32 (patch size)

# Slicing up to len(actuals) guarantees we don't accidentally pass extra rows 
# if the notebook memory state gets messed up again
short_context_data = [series[-SHORT_CONTEXT_LEN:] for series in context_data][:len(actuals)]

# Re-compile a dedicated model for the shorter context
tfm_model_short = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
tfm_model_short.compile(
    timesfm.ForecastConfig(
        max_context=SHORT_CONTEXT_LEN,
        max_horizon=CV_FOLD_HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
        per_core_batch_size=32  # Lowering this helps prevent truncation
    )
)

with mlflow.start_run(run_name="TimesFM_ZeroShot_ShortContext"):
    mlflow.set_tags({"model_type": "TimesFM", "run_type": "zero-shot-eval", "variant": "short_context"})

    start_time = time.time()
    
    # ---------------------------------------------------------
    # THE FIX: Manual Chunking to bypass TimesFM batching bug
    # ---------------------------------------------------------
    CHUNK_SIZE = 500
    all_point_fcs = []
    
    print("Forecasting in chunks to prevent row-dropping...")
    for i in range(0, len(short_context_data), CHUNK_SIZE):
        chunk = short_context_data[i : i + CHUNK_SIZE]
        p_fc, _ = tfm_model_short.forecast(horizon=CV_FOLD_HORIZON, inputs=chunk)
        all_point_fcs.append(p_fc)
        
    # Combine the chunks back together
    point_fc = np.concatenate(all_point_fcs, axis=0)
    preds = np.clip(point_fc, 0, None)
    elapsed = time.time() - start_time

    # Defensive check
    if preds.shape[0] != len(short_context_data):
        raise ValueError(
            f"forecast() returned {preds.shape[0]} rows for {len(short_context_data)} "
            "input series - mismatch likely due to internal batching/packing. "
        )

    # Calculate WMAE
    abs_err = np.abs(preds - np.array(actuals))
    wmae_short = np.sum(np.array(weights) * abs_err) / np.sum(np.array(weights))

    mlflow.log_metric("val_wmae", wmae_short)
    mlflow.log_metric("inference_time_minutes", elapsed / 60)
    mlflow.log_param("context_len", SHORT_CONTEXT_LEN)
    mlflow.log_param("horizon", CV_FOLD_HORIZON)
    mlflow.log_param("normalize_inputs", True)
    mlflow.log_param("use_continuous_quantile_head", True)

    print(f"TimesFM Short-Context ({SHORT_CONTEXT_LEN}w) WMAE: {wmae_short:.2f}")

Forecasting in chunks to prevent row-dropping...
TimesFM Short-Context (64w) WMAE: 1483.06
🏃 View run TimesFM_ZeroShot_ShortContext at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12/runs/91bbe622b1db4548bb4f4d1b9fc844f3
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12


## Run 3: Continuous Quantile Head On vs. Off

In [16]:
tfm_model_noq = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
tfm_model_noq.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT_LEN,
        max_horizon=CV_FOLD_HORIZON,
        normalize_inputs=True,
        use_continuous_quantile_head=False,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

with mlflow.start_run(run_name="TimesFM_ZeroShot_NoQuantileHead"):
    mlflow.set_tags({"model_type": "TimesFM", "run_type": "zero-shot-eval", "variant": "no_quantile_head"})

    # SAFETY FIX: Slice the context data to match the exact length of actuals (3331)
    # in case the notebook session state got duplicated or out of sync
    context_data_clean = context_data[:len(actuals)]

    start_time = time.time()
    
    # ---------------------------------------------------------
    # Manual Chunking for Run 3 using cleaned context data
    # ---------------------------------------------------------
    CHUNK_SIZE = 500
    all_point_fcs = []
    
    print("Forecasting in chunks to prevent row-dropping...")
    for i in range(0, len(context_data_clean), CHUNK_SIZE):
        chunk = context_data_clean[i : i + CHUNK_SIZE]
        p_fc, _ = tfm_model_noq.forecast(horizon=CV_FOLD_HORIZON, inputs=chunk)
        all_point_fcs.append(p_fc)
        
    point_fc = np.concatenate(all_point_fcs, axis=0)
    preds = np.clip(point_fc, 0, None)
    elapsed = time.time() - start_time

    if preds.shape[0] != len(context_data_clean):
        raise ValueError(
            f"forecast() returned {preds.shape[0]} rows for {len(context_data_clean)} input series."
        )

    # This will now broadcast perfectly (3331, 13) vs (3331, 13)
    abs_err = np.abs(preds - np.array(actuals))
    wmae_noq = np.sum(np.array(weights) * abs_err) / np.sum(np.array(weights))

    mlflow.log_metric("val_wmae", wmae_noq)
    mlflow.log_metric("inference_time_minutes", elapsed / 60)
    mlflow.log_param("context_len", CONTEXT_LEN)
    mlflow.log_param("horizon", CV_FOLD_HORIZON)
    mlflow.log_param("use_continuous_quantile_head", False)

    print(f"TimesFM (no quantile head) WMAE: {wmae_noq:.2f}")
    print(f"vs. quantile-head-on WMAE: {wmae:.2f}")

Forecasting in chunks to prevent row-dropping...
TimesFM (no quantile head) WMAE: 1156.72
vs. quantile-head-on WMAE: 1156.72
🏃 View run TimesFM_ZeroShot_NoQuantileHead at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12/runs/49648ec5988448cc869fa9b2f0d7d603
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12


## Run 4: Adding Holiday as a Dynamic Covariate (XReg)

In [14]:
holiday_context_and_horizon = [h[:val_start_week + CV_FOLD_HORIZON].astype(str) for h in holiday_full]

with mlflow.start_run(run_name="TimesFM_ZeroShot_HolidayCovariate"):
    mlflow.set_tags({"model_type": "TimesFM", "run_type": "zero-shot-eval", "variant": "holiday_covariate"})

    start_time = time.time()
    try:

        CHUNK_SIZE = 500
        all_point_fcs = []
        
        print("Forecasting with covariates in chunks to prevent row-dropping...")
        for i in range(0, len(context_data), CHUNK_SIZE):
            chunk_inputs = context_data[i : i + CHUNK_SIZE]
            chunk_holidays = holiday_context_and_horizon[i : i + CHUNK_SIZE]
            
            p_fc, _ = tfm_model.forecast_with_covariates(
                inputs=chunk_inputs,
                dynamic_categorical_covariates={"is_holiday": chunk_holidays},
                xreg_mode="xreg + timesfm",
            )
            all_point_fcs.append(p_fc)
            
        point_fc = np.concatenate(all_point_fcs, axis=0)
        preds = np.clip(point_fc, 0, None)
        elapsed = time.time() - start_time

        # Defensive check
        if preds.shape[0] != len(context_data):
            raise ValueError(f"forecast_with_covariates() returned {preds.shape[0]} rows for {len(context_data)} input series.")

        abs_err = np.abs(preds - np.array(actuals))
        wmae_cov = np.sum(np.array(weights) * abs_err) / np.sum(np.array(weights))

        mlflow.log_metric("val_wmae", wmae_cov)
        mlflow.log_metric("inference_time_minutes", elapsed / 60)
        mlflow.log_param("context_len", CONTEXT_LEN)
        mlflow.log_param("horizon", CV_FOLD_HORIZON)
        mlflow.log_param("xreg_mode", "xreg + timesfm")

        print(f"TimesFM + holiday covariate WMAE: {wmae_cov:.2f}")
        print(f"vs. zero-shot (no covariate) WMAE: {wmae:.2f}")
        
    except Exception as e:
        mlflow.set_tag("status", "failed")
        mlflow.log_param("error", str(e)[:250])
        print(f"Covariate run failed: {e}")

Forecasting with covariates in chunks to prevent row-dropping...
Covariate run failed: For XReg, `return_backcast` must be set to True in the forecast config. Please recompile the model.
🏃 View run TimesFM_ZeroShot_HolidayCovariate at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12/runs/f69fd02809a443958d3f5b24b6c02134
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12


## Comparison Summary

In [17]:
test_fe = pd.read_parquet(BASE + 'test_features.parquet')
test_fe["Date"] = pd.to_datetime(test_fe["Date"])

cold_start_pairs = list(set(zip(test_fe.Store, test_fe.Dept)) - set(zip(train_fe.Store, train_fe.Dept)))
dept_avg_fallback = train_fe.groupby('Dept')['Weekly_Sales'].mean()

test_dates = pd.date_range(full_date_range.max() + pd.Timedelta(days=7), periods=39, freq='7D')

## Final Pipeline

In [18]:
class TimesFMPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, cold_start_pairs, dept_avg_fallback,
                 full_date_range, test_dates, max_context=128, horizon=39):
        self.cold_start_pairs = cold_start_pairs
        self.dept_avg_fallback = dept_avg_fallback
        self._fallback_mean = dept_avg_fallback.mean()  
        self.full_date_range = full_date_range
        self.test_dates = test_dates
        self.max_context = max_context
        self.horizon = horizon
        self._model = None

    def _load_model(self):
        if self._model is None:
            self._model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
                "google/timesfm-2.5-200m-pytorch"
            )
            self._model.compile(timesfm.ForecastConfig(
                max_context=self.max_context,
                max_horizon=self.horizon,
                normalize_inputs=True,
                use_continuous_quantile_head=True,
                force_flip_invariance=True,
                infer_is_positive=True,
                fix_quantile_crossing=True,
            ))
        return self._model

    def predict(self, context, model_input):
        model = self._load_model()
        active_pairs = sorted(set(zip(model_input.Store, model_input.Dept)))

        context_data = []
        for store, dept in active_pairs:
            series = model_input[(model_input.Store == store) & (model_input.Dept == dept)][['Date', 'Weekly_Sales']]
            series = series.set_index('Date').reindex(self.full_date_range)
            sales = series['Weekly_Sales'].fillna(0).clip(lower=0).values.astype(np.float32)
            context_data.append(sales[-self.max_context:])
        
        CHUNK_SIZE = 500
        all_point_fcs = []
        for i in range(0, len(context_data), CHUNK_SIZE):
            chunk = context_data[i : i + CHUNK_SIZE]
            p_fc, _ = model.forecast(horizon=self.horizon, inputs=chunk)
            all_point_fcs.append(p_fc)
            
        point_fc = np.concatenate(all_point_fcs, axis=0)
        preds = np.clip(point_fc, 0, None)

        rows = []
        for i, (store, dept) in enumerate(active_pairs):
            for j, date in enumerate(self.test_dates):
                rows.append({'Store': store, 'Dept': dept, 'Date': date, 'Weekly_Sales': preds[i, j]})
        
        for store, dept in self.cold_start_pairs:
            fallback_val = self.dept_avg_fallback.get(dept, self._fallback_mean)
            for date in self.test_dates:
                rows.append({'Store': store, 'Dept': dept, 'Date': date, 'Weekly_Sales': fallback_val})

        return pd.DataFrame(rows)

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [19]:
example_pairs = active_pairs[:2]  
example_input = train_fe[
    train_fe.apply(lambda r: (r.Store, r.Dept) in example_pairs, axis=1)
][['Store', 'Dept', 'Date', 'Weekly_Sales']]

with mlflow.start_run(run_name="TimesFM_Pipeline_Log"):
    mlflow.pyfunc.log_model(
        python_model=TimesFMPipeline(
            cold_start_pairs=cold_start_pairs,
            dept_avg_fallback=dept_avg_fallback,
            full_date_range=full_date_range,
            test_dates=test_dates,
            max_context=128,  
            horizon=39
        ),
        artifact_path="model",
        input_example=example_input,
        registered_model_name="walmart_timesfm_pipeline"  
    )
    print("TimesFM registered as a pipeline")

2026/07/12 11:38:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 11:38:53 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/12 11:38:53 INFO mlflow.pyfunc: Inferring model signature from input example
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training

TimesFM registered as a pipeline
🏃 View run TimesFM_Pipeline_Log at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12/runs/e882aa54c81a435bac8c74cae087a050
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/12
